<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/Mia/tests_rev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import json
from plotly.subplots import make_subplots
from shapely.geometry import Polygon

In [5]:
import requests
import pandas as pd
import geopandas as gpd
from io import BytesIO

url = "https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_businesses_nhood_naics.parquet"

response = requests.get(url)
response.raise_for_status()

# STEP 1: read as pandas dataframe
sf_map = pd.read_parquet(BytesIO(response.content))

sf_map.head()

,nhood,naics_group,year,closed,opened,biz_stock
0,Western Addition,Manufacturing & Industrial,2016,8.0,24.0,168
1,Western Addition,Utilities & Construction,2016,4.0,7.0,168
2,Western Addition,Retail,2016,5.0,23.0,168
3,Western Addition,Personal Services,2016,3.0,4.0,168
4,Western Addition,Food & Entertainment,2016,4.0,15.0,168


In [6]:
sf_map.columns

Index(['nhood', 'naics_group', 'year', 'closed', 'opened', 'biz_stock'], dtype='object')

In [7]:
# STEP 2: Clean column types

sf_map['year'] = sf_map['year'].astype(int)
sf_map['opened'] = sf_map['opened'].fillna(0).astype(int)
sf_map['closed'] = sf_map['closed'].fillna(0).astype(int)
sf_map['biz_stock'] = sf_map['biz_stock'].fillna(0).astype(int)

sf_map['closed_negative'] = -sf_map['closed']
sf_map['net'] = sf_map['opened'] - sf_map['closed']

In [8]:
# STEP 3: Summarize by neighborhood and year across all NAICS groups

sf_neigh_year = (
    sf_map
    .groupby(['nhood', 'year'])[['opened', 'closed', 'biz_stock']]
    .sum()
    .reset_index()
)

sf_neigh_year['closed_negative'] = -sf_neigh_year['closed']
sf_neigh_year['net'] = sf_neigh_year['opened'] - sf_neigh_year['closed']

In [9]:
# STEP 4: Create all-neighborhood totals

sf_all_year = (
    sf_neigh_year
    .groupby('year')[['opened', 'closed', 'biz_stock']]
    .sum()
    .reset_index()
)

sf_all_year['nhood'] = 'All neighborhoods'
sf_all_year['closed_negative'] = -sf_all_year['closed']
sf_all_year['net'] = sf_all_year['opened'] - sf_all_year['closed']

In [10]:
# STEP 5: Combine all-neighborhood and neighborhood-level data

sf_neigh_year_dropdown = pd.concat(
    [sf_all_year, sf_neigh_year],
    ignore_index=True
)

In [11]:
# STEP 6: Fill missing neighborhood-year combinations

years = sorted(sf_neigh_year_dropdown['year'].dropna().unique())

sf_dropdown_neighborhoods = ['All neighborhoods'] + sorted(
    [
        n for n in sf_neigh_year_dropdown['nhood'].dropna().unique()
        if n != 'All neighborhoods'
    ]
)

sf_neigh_dropdown_complete = (
    pd.MultiIndex.from_product(
        [sf_dropdown_neighborhoods, years],
        names=['nhood', 'year']
    )
    .to_frame(index=False)
    .merge(
        sf_neigh_year_dropdown,
        on=['nhood', 'year'],
        how='left'
    )
)

sf_neigh_dropdown_complete[
    ['opened', 'closed', 'biz_stock', 'closed_negative', 'net']
] = sf_neigh_dropdown_complete[
    ['opened', 'closed', 'biz_stock', 'closed_negative', 'net']
].fillna(0)

In [12]:
# STEP 7: Create fixed y-axis ranges

sf_individual_neighs = sf_neigh_dropdown_complete[
    sf_neigh_dropdown_complete['nhood'] != 'All neighborhoods'
]

sf_individual_max = sf_individual_neighs[['opened', 'closed']].to_numpy().max()
sf_individual_range = [-sf_individual_max * 1.1, sf_individual_max * 1.1]

sf_all_neighs = sf_neigh_dropdown_complete[
    sf_neigh_dropdown_complete['nhood'] == 'All neighborhoods'
]

sf_all_max = sf_all_neighs[['opened', 'closed']].to_numpy().max()
sf_all_range = [-sf_all_max * 1.1, sf_all_max * 1.1]

In [13]:
# STEP 8: Build dropdown chart

fig = go.Figure()

for nhood in sf_dropdown_neighborhoods:
    n_df = sf_neigh_dropdown_complete[
        sf_neigh_dropdown_complete['nhood'] == nhood
    ]

    fig.add_trace(go.Bar(
        x=n_df['year'],
        y=n_df['opened'],
        name='Opened',
        visible=(nhood == 'All neighborhoods'),
        showlegend=(nhood == 'All neighborhoods')
    ))

    fig.add_trace(go.Bar(
        x=n_df['year'],
        y=n_df['closed_negative'],
        name='Closed',
        visible=(nhood == 'All neighborhoods'),
        showlegend=(nhood == 'All neighborhoods')
    ))

buttons = []

for i, nhood in enumerate(sf_dropdown_neighborhoods):
    visible = [False] * (len(sf_dropdown_neighborhoods) * 2)
    visible[i * 2] = True
    visible[i * 2 + 1] = True

    y_range = sf_all_range if nhood == 'All neighborhoods' else sf_individual_range

    buttons.append(dict(
        label=nhood,
        method='update',
        args=[
            {'visible': visible},
            {
                'title': f'Business Openings and Closings in {nhood}',
                'yaxis': {'range': y_range}
            }
        ]
    ))

fig.update_layout(
    title='Business Openings and Closings in All neighborhoods',
    xaxis_title='Year',
    yaxis_title='Number of Businesses',
    yaxis=dict(range=sf_all_range),
    barmode='relative',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=1.05,
        y=1,
        showactive=True
    )],
    legend_title_text='Business Status'
)

fig.add_hline(y=0)

fig.show()

# Bubble

In [18]:
max_openings = (
    sf_neigh_dropdown_complete[
        sf_neigh_dropdown_complete['nhood'] != 'All neighborhoods'
    ]
    .groupby('nhood')['opened']
    .max()
)

eligible_neighborhoods = max_openings.sort_values(ascending=False).head(10).index.tolist()

In [22]:
sf_centroids = sf_map.copy()

# Project before calculating centroids
sf_centroids_projected = sf_centroids.to_crs(epsg=7131)
sf_centroids['centroid'] = sf_centroids_projected.geometry.centroid.to_crs(epsg=4326)

sf_centroids['lon'] = sf_centroids['centroid'].x
sf_centroids['lat'] = sf_centroids['centroid'].y

AttributeError: 'DataFrame' object has no attribute 'to_crs'